# Transects

Casts shore-normal transects along the OS-derived midline reference shoreline, for use with `SDS_transects.compute_intersection_QC`.

**Parameters decided here**, each from a diagnostic rather than a default:
- `window` — the chord half-width for `normal_direction`, chosen from where transect crossings stop being scattered noise and become real curvature.
- `landward` / `seaward` — transect extent, sized from the measured intertidal width plus the erosion envelope over the study period.
- `end_trim` — how much of the northern end to drop, from the crossing analysis.

**Verified here:** that transects point seaward (`verify_seaward`), and that none cross within the measurement zone.

**Produces:** `transects_holderness.geojson` at 50 m spacing, plus a metadata JSON. Decimation happens at use time — `sorted(transects)[::5]` gives 250 m spacing from the same file, so it never needs regenerating.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString

from holderness import config, reference
from holderness import transects as T

config.OUT.mkdir(parents=True, exist_ok=True)

config.FIGS.mkdir(parents=True, exist_ok=True)

def savefig(fig, name):
    """Save a figure to outputs/figures with consistent settings."""
    for ext in ('png', 'pdf'):
        fig.savefig(config.FIGS / f'{name}.{ext}', dpi=config.FIG_DPI,
                    bbox_inches='tight')
        
print('repo :', config.REPO_ROOT)
print('gml  :', config.GML_PATH, '(exists)' if config.GML_PATH.exists() else '(MISSING)')
print('out  :', config.OUT)

In [ ]:
# Load the reference midline and run hairpin_locations to get
# midline coordinate hairpins

midline_arr = reference.load_reference(config.OUT / 'refsl_holderness_os_midline.pkl')
midline = LineString(midline_arr)
midline_hairpins = reference.hairpin_locations(midline)

In [ ]:
print(midline_hairpins)

In [ ]:
mlw = LineString(reference.load_reference(config.OUT / 'refsl_holderness_os_mlw.pkl'))

# Chose window

In [ ]:
results = {}
for w in [50, 100, 250, 500]:
    tr, d = T.build_transects(midline, landward=config.TRANSECT_LANDWARD, seaward=config.TRANSECT_SEAWARD, window=w,
                              skip_ranges=midline_hairpins, pad=w)
    results[w] = {
        'transects': tr,
        'd_along': d,
        'n': len(tr),
        'crossings': len(T.check_crossings(tr)),
        'seaward_fail': len(T.verify_seaward(tr, mlw)),
    }
    print(f"window {w:4d}: {results[w]['n']:5d} transects, "
          f"{results[w]['crossings']:3d} crossings, "
          f"{results[w]['seaward_fail']} seaward failures")

In [ ]:
def bearings(tr):
    names = sorted(tr)
    v = np.array([tr[n][1] - tr[n][0] for n in names])
    return np.degrees(np.arctan2(v[:, 0], v[:, 1]))   # from north

for w, r in results.items():
    b = bearings(r['transects'])
    print(f'window {w:4d}: median |Δbearing| between neighbours '
          f'{np.median(np.abs(np.diff(b))):.2f}°')

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7))

for i, (w, r) in enumerate(results.items()):
    b = bearings(r['transects'])
    for ax in (ax1, ax2):
        ax.plot(r['d_along'] / 1000, b, color=f'C{i}', lw=0.7,
                label=f'window {w} m')

ax1.set_ylabel('transect bearing (°from N)')
ax1.set_title('full reach')

ax2.set_xlim(24, 28)          # 4 km window to see the jitter
ax2.set_ylabel('bearing (°from N)')
ax2.set_xlabel('alongshore distance from Kilnsea (km)')
ax2.set_title('detail')

for ax in (ax1, ax2):
    ax.grid(ls=':', color='0.8')
ax1.legend(fontsize=8, ncol=4)
fig.tight_layout()
savefig(fig, 'transect_window_bearing')

In [ ]:
# Extended results
ext_results = {}
for w in [50, 100, 250, 500, 1000, 1500, 2000]:
    tr, d = T.build_transects(midline, landward=config.TRANSECT_LANDWARD, seaward=config.TRANSECT_SEAWARD,
                              window=w, skip_ranges=midline_hairpins, pad=w)
    ext_results[w] = {
        'transects': tr,
        'd_along': d,
        'n': len(tr),
        'crossings': len(T.check_crossings(tr)),
        'seaward_fail': len(T.verify_seaward(tr, mlw)),
    }
    print(f"window {w:4d}: {ext_results[w]['n']:5d} transects, "
          f"{ext_results[w]['crossings']:3d} crossings, "
          f"{ext_results[w]['seaward_fail']} seaward failures")

for w, r in ext_results.items():
    b = bearings(r['transects'])
    print(f'window {w:4d}: median |Δbearing| between neighbours '
          f'{np.median(np.abs(np.diff(b))):.2f}°')

### Why bearing jitter cannot choose the window

Jitter halves with every doubling of the window, all the way out to 2000 m — 1.42° → 0.79° → 0.38° → 0.22°. That exact 1/w scaling is the signature of averaging *noise*: if real coastal curvature were being resolved, jitter would stop falling once the window reached the curvature scale. It never plateaus, so the metric only ever says "larger is smoother", which is not a criterion.

It also doesn't matter. Chainage error from an angular offset θ scales as 1/cos θ, so even 1.42° costs 7.7 cm over a 250 m transect — and being random, it averages out across transects rather than biasing the mean. Angular noise is not a reason to widen the window.

The cost that does matter is crossings (below), and the cost of over-correcting is smoothing away real curvature.

In [ ]:
# Chainage error from angular jitter: measured distance is inflated by 1/cos(theta)

for w, r in results.items():
    j = np.median(np.abs(np.diff(bearings(r['transects']))))
    err = 1 / np.cos(np.radians(j)) - 1
    print(f'window {w:4d}: jitter {j:.2f} deg -> {err*100:.4f}% '
          f'= {err * config.TRANSECT_SEAWARD * 100:.1f} cm over the seaward extent')

## Crossing locations

In [ ]:
for w, r in results.items():
    tr, da = r['transects'], r['d_along']
    idx = {n: i for i, n in enumerate(sorted(tr))}
    cr = T.check_crossings(tr)
    if cr:
        locs = np.array([da[idx[a]] for a, b in cr]) / 1000
        print(f'window {w:4d}: {len(cr):3d} crossings, '
              f'alongshore {locs.min():.1f}-{locs.max():.1f} km')
    else:
        print(f'window {w:4d}:   0 crossings')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))

for row, (w, r) in enumerate(results.items()):
    tr, da = r['transects'], r['d_along']
    pos = {n: k for k, n in enumerate(sorted(tr))}
    cr = T.check_crossings(tr)
    locs = np.array([da[pos[a]] for a, b in cr]) / 1000 if cr else np.array([])
    ax.plot(locs, np.full(len(locs), row), '|', ms=12,
            color=f'C{row}', label=f'window {w} m ({len(cr)})')

ax.set_yticks(range(len(results)))
ax.set_yticklabels([f'{w} m' for w in results])
ax.set_xlabel('alongshore distance from Kilnsea (km)')
ax.set_xlim(0, midline.length / 1000)
ax.set_title('transect crossing locations by window')
ax.grid(axis='x', ls=':', color='0.8')
fig.tight_layout()
savefig(fig, 'transect_window_crossings')

## Cost of over-smoothing

In [ ]:
ref = T.build_transects(midline, landward=config.TRANSECT_LANDWARD, seaward=config.TRANSECT_SEAWARD,
                        window=100, skip_ranges=midline_hairpins, pad=1000)[0]
b_ref = bearings(ref)

for w in [250, 500, 1000, 2000]:
    tr, da = T.build_transects(midline, landward=config.TRANSECT_LANDWARD, seaward=config.TRANSECT_SEAWARD,
                               window=w, skip_ranges=midline_hairpins, pad=1000)
    dev = np.abs(bearings(tr) - b_ref)
    bay = da > 55000
    print(f'window {w:4d}: median deviation in Bridlington Bay {np.median(dev[bay]):.2f} deg')

Deviation from the w=100 baseline in Bridlington Bay, where the coast genuinely curves: 0.49° at 250 m, 0.62° at 500 m, 1.61° at 1000 m, 2.50° at 2000 m.

Only the curved section measures real bias — in the straight reach the w=100 baseline is itself noise-dominated, so deviation there measures the baseline's noise rather than the wider window's error.

**Conclusion: 250 m.** It is where crossings stop being scattered along the whole reach (noise) and become confined to the final 800 m (Bridlington Bay curvature), while costing only 0.49° of smoothing bias. Going to 500 m removes the remaining crossings but doubles the bias — using a global smoothing parameter to fix a localised geometric problem. Those crossings are better handled by trimming the northern end.

## Landward and seaward

In [ ]:
# Envelope

_s = np.load(config.OUT / f'separation_{config.SITE}.npz')
d_sep, sep = _s['along'], _s['separation']
print(f'median intertidal width {np.median(sep):.0f} m')

YEARS = 34
for rate in [1.2, 2.0, 3.0, 5.0]:
    print(f'{rate} m/yr -> {rate*YEARS:.0f} m retreat over {YEARS} years')
print(f'\nplus half the median intertidal width: {np.median(sep)/2:.0f} m')
print(f'seaward extent needed = retreat + tidal excursion + storm margin')

### Sizing the extents

The required extent is the erosion envelope plus tidal excursion plus margin - but the erosion envelope is measured *relative to the OS line*, whose survey date is unknown. OS publish no capture date for `TidalBoundary` and state that tide lines are not revised frequently, so the line may predate the study period by decades.

That uncertainty is symmetric. Shorelines from before the OS survey lie seaward of it; shorelines from after lie landward. With the date unknown, both extents must accommodate the full envelope:

- tidal excursion: ±57 m (half the median intertidal width)
- retreat over 34 years: 41 m at 1.2 m/yr, 68 m at 2.0, 102 m at 3.0, 170 m at 5.0
- unknown OS offset: unbounded, plausibly 60–150 m either way
- storm and detection noise: tens of metres

Set generously: `TRANSECT_LANDWARD = 300`, `TRANSECT_SEAWARD = 400`. An over-long transect costs only a wider search; a short one silently truncates real shorelines exactly where erosion is fastest, and the loss would be invisible in the output.

**To revisit after first extraction:** overlay 1990 and 2024 shorelines on the transects and confirm both fall well inside the extents. This also dates the OS line empirically — the year at which the cross-shore offset crosses zero.

## Decisions

| parameter | value | justification |
|---|---|---|
| `TRANSECT_SPACING` | 50 m | Above the ~50 m floor set by `along_dist=25` (closer transects share detected points) and by Landsat's 30 m pixel. Fine enough to decimate later - `sorted(transects)[::5]` gives 250 m without regenerating. |
| `TRANSECT_WINDOW` | 250 m | Where crossings stop being scattered along the reach (noise) and become confined to Bridlington Bay (real curvature), at only 0.49° of smoothing bias. Bearing jitter cannot decide this - it falls as 1/w indefinitely and costs under 1 cm of chainage error. |
| `TRANSECT_LANDWARD` | 300 m | Tidal excursion (57 m) plus the landward offset of post-survey shorelines from an OS line of unknown date. Generous because that offset is unbounded. |
| `TRANSECT_SEAWARD` | 400 m | Retreat envelope over 34 years (68–170 m depending on rate) plus tidal excursion plus the seaward offset of pre-survey shorelines. To be verified against extracted shorelines. |
| `TRANSECT_END_TRIM` | 500 m | Removes all 11 remaining crossings at `window=250`, which occur 114–367 m from the origin - inside the measurement zone, not safely offshore. Costs 0.8% of the reach at the northern terminus. Justified geometrically, not by selecting on coastal character.

## Build

In [ ]:
tr_all, d_all = T.build_transects(
    midline,
    spacing=config.TRANSECT_SPACING,
    landward=config.TRANSECT_LANDWARD,
    seaward=config.TRANSECT_SEAWARD,
    window=config.TRANSECT_WINDOW,
    skip_ranges=midline_hairpins,
)

keep = d_all <= midline.length - config.TRANSECT_END_TRIM
transects = {n: tr_all[n] for n, k in zip(sorted(tr_all), keep) if k}
d_along = d_all[keep]

print(f'{len(tr_all)} built, {len(transects)} retained after end trim')
print('crossings:', len(T.check_crossings(transects)))
print('seaward failures:', len(T.verify_seaward(transects, mlw)))

## Plot

In [ ]:
tarr = np.array([transects[n] for n in sorted(transects)])
mid_c = np.array(midline.coords)
mlw_c = np.array(mlw.coords)

fig, axes = plt.subplots(1, 3, figsize=(15, 6))

for ax, (title, along, half) in zip(axes, [
    ('full reach', None, None),
    ('mid-reach detail', 30_000, 700),
    ('Bridlington end', 58_000, 700),
]):
    ax.plot(*mid_c.T, color='C2', lw=0.8, label='midline', zorder=3)
    ax.plot(*mlw_c.T, color='C0', lw=0.8, label='MLW', zorder=3)
    step = 20 if along is None else 1
    for t in tarr[::step]:
        ax.plot(t[:, 0], t[:, 1], color='0.5', lw=0.4, zorder=1)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xlabel('Easting (m)')
    if along is not None:
        p = midline.interpolate(along)
        ax.set_xlim(p.x - half, p.x + half)
        ax.set_ylim(p.y - half, p.y + half)

axes[0].set_ylabel('Northing (m)')
axes[0].legend(fontsize=8)
fig.tight_layout()
savefig(fig, 'transects_overview')

In [ ]:
# Classify as defended coastline or not

labels_arr = T.classify_defence(d_along, config.DEFENDED, config.DOWNDRIFT_M)
labels = dict(zip(sorted(transects), labels_arr))

import collections
print(collections.Counter(labels.values()))

T.transects_to_geojson(transects, config.OUT / f'transects_{config.SITE}.geojson',
                       config.EPSG, labels=labels)

## Save

In [ ]:
T.transects_to_geojson(transects, config.OUT / f'transects_{config.SITE}.geojson', config.EPSG)

meta = {
    'site': config.SITE,
    'source_line': 'refsl_holderness_os_midline.pkl',
    'spacing_m': config.TRANSECT_SPACING,
    'window_m': config.TRANSECT_WINDOW,
    'landward_m': config.TRANSECT_LANDWARD,
    'seaward_m': config.TRANSECT_SEAWARD,
    'end_trim_m': config.TRANSECT_END_TRIM,
    'skipped_hairpins': midline_hairpins,
    'n_built': len(tr_all),
    'n_retained': len(transects),
    'crossings': T.check_crossings(transects),
    'seaward_failures': T.verify_seaward(transects, mlw),
    'os_survey_date': None,
    'os_survey_date_note': (
        'Not published for OS OpenMap Local; TidalBoundary carries no capture date.' 
        'OS state tide lines are not revised frequently, so the line may predate the'
        'study period by decades. Effective date to be estimated empirically from the'
        'satellite record.'
    ),
    'median_intertidal_width_m': round(float(np.median(sep)), 1),
    'crossings_before_trim': len(T.check_crossings(tr_all)),
}
with open(config.OUT / f'transects_meta_{config.SITE}.json', 'w') as f:
    json.dump(meta, f, indent=2)